# Continuous walk-forward deployment

This notebook simulates one account while Cobasket reselects a strategy in successive folds. Unlike the repeated-fold summary, cash and holdings are not reset between test intervals.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from cobasket.continuous_walk_forward import (
    ContinuousDeploymentConfig,
    run_continuous_walk_forward,
)
from cobasket.repeated_walk_forward import WalkForwardConfig
from cobasket.strategy_experiments import StrategyExperimentConfig
from cobasket.strategy_rules import MetricCondition, StrategyRule, StrategyRules

## Synthetic historical data

The example uses two assets with changing return regimes and a precomputed probability metric. In a real Cobasket workflow, the probability table would come from the leakage-safe evidence and calibration pipeline.

In [ ]:
rng = np.random.default_rng(42)
dates = pd.date_range('2017-01-02', periods=1000, freq='B')
regime = np.where(np.arange(len(dates)) < 500, 1.0, -0.4)
returns_a = 0.0004 + 0.0005 * regime + rng.normal(0, 0.012, len(dates))
returns_b = 0.0003 - 0.0002 * regime + rng.normal(0, 0.010, len(dates))
prices = pd.DataFrame({
    'AAA': 100 * np.exp(np.cumsum(returns_a)),
    'BBB': 100 * np.exp(np.cumsum(returns_b)),
}, index=dates)
probability = pd.DataFrame({
    'AAA': np.where(regime > 0, 0.70, 0.40),
    'BBB': np.where(regime > 0, 0.45, 0.68),
}, index=dates)
metrics = {'probability': probability}

## Candidate strategies

The candidates differ only in how demanding their buy threshold is. Each fold chooses between them using validation data only.

In [ ]:
def probability_strategy(name, buy_threshold):
    return StrategyRules(
        name=name,
        rules=(
            StrategyRule('sell', (MetricCondition('probability', '<=', 0.35),), 0.0),
            StrategyRule('buy', (MetricCondition('probability', '>=', buy_threshold),), 0.45),
        ),
        transaction_cost_bps=10.0,
    )

strategies = (
    probability_strategy('moderate threshold', 0.60),
    probability_strategy('strict threshold', 0.68),
)

In [ ]:
walk_forward = WalkForwardConfig(
    train_observations=300,
    validation_observations=100,
    test_observations=100,
    step_observations=100,
)
experiment = StrategyExperimentConfig(
    selection_metric='sharpe_ratio',
    initial_cash=10_000.0,
)
result = run_continuous_walk_forward(
    prices,
    metrics,
    strategies,
    walk_forward=walk_forward,
    experiment=experiment,
    deployment=ContinuousDeploymentConfig(
        initial_cash=10_000.0,
        boundary_policy='retain',
    ),
)

In [ ]:
result.selections

In [ ]:
result.metrics

In [ ]:
result.equity.plot(figsize=(11, 5), title='Continuous walk-forward account')
plt.ylabel('Portfolio value')
plt.show()

## Retain versus liquidate

`retain` leaves existing holdings in place when the selected strategy changes. `liquidate` closes the account at that boundary and lets the new strategy re-enter later. Comparing both is a useful turnover and transaction-cost sensitivity test.

In [ ]:
liquidated = run_continuous_walk_forward(
    prices, metrics, strategies,
    walk_forward=walk_forward,
    experiment=experiment,
    deployment=ContinuousDeploymentConfig(
        initial_cash=10_000.0,
        boundary_policy='liquidate',
    ),
)
pd.DataFrame({
    'retain': result.metrics.loc['strategy'],
    'liquidate': liquidated.metrics.loc['strategy'],
})

A credible strategy should not rely on one exact boundary convention. Large differences between the two policies indicate that strategy reselection and turnover are important parts of the apparent result.